In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import xgboost as xgb
import joblib
import json
import os
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")


In [ ]:
DATA_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data"
MODELS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models"
METRICS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\metrics"
EXPERIMENTS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\experiments"
DOCS_PATH = "C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\docs"
RANDOM_SEED = 42

for path in [MODELS_PATH, METRICS_PATH, EXPERIMENTS_PATH, DOCS_PATH]:
    os.makedirs(path, exist_ok=True)


In [4]:
print("Loading preprocessing pipeline...")
preprocessor = joblib.load("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\models\\processingpreprocessing_pipeline.pkl")
print("✓ Preprocessor loaded")

print("\nLoading datasets...")

X_train_raw = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_train.csv")
X_val_raw   = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_val.csv")
X_test_raw  = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data X_test.csv")

y_cost_train = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_cost_train.csv").values.ravel()
y_cost_val   = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_cost_val.csv").values.ravel()
y_cost_test  = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_cost_test.csv").values.ravel()

y_co2_train = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_train.csv").values.ravel()
y_co2_val   = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_val.csv").values.ravel()
y_co2_test  = pd.read_csv("C:\\Users\\sneha\\Desktop\\ecopackai\\Data\\ml\\ml_data\\ml_data y_co2_test.csv").values.ravel()

print("✓ Data loaded successfully")


Loading preprocessing pipeline...
✓ Preprocessor loaded

Loading datasets...
✓ Data loaded successfully


In [5]:
X_train = preprocessor.transform(X_train_raw)
X_val   = preprocessor.transform(X_val_raw)
X_test  = preprocessor.transform(X_test_raw)

print("Transformed shapes:")
print("Train:", X_train.shape)
print("Val  :", X_val.shape)
print("Test :", X_test.shape)


Transformed shapes:
Train: (424200, 23)
Val  : (60600, 23)
Test : (121200, 23)


In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0, random_state=RANDOM_SEED),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=RANDOM_SEED),
    "Random Forest": RandomForestRegressor(
        n_estimators=50, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1
    ),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=50, max_depth=6, learning_rate=0.1, random_state=RANDOM_SEED
    )
}

print("Models defined:")
for m in models:
    print(" -", m)


Models defined:
 - Linear Regression
 - Ridge Regression
 - Decision Tree
 - Random Forest
 - XGBoost


In [7]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
print("✓ 5-Fold Cross-Validation ready")


✓ 5-Fold Cross-Validation ready


In [8]:
def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test):
    model.fit(X_train, y_train)
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2", n_jobs=-1)
    
    metrics = {
        "cv_r2_mean": float(cv_scores.mean()),
        "cv_r2_std": float(cv_scores.std()),
        "train_mae": float(mean_absolute_error(y_train, model.predict(X_train))),
        "train_rmse": float(np.sqrt(mean_squared_error(y_train, model.predict(X_train)))),
        "train_r2": float(r2_score(y_train, model.predict(X_train))),
        "val_mae": float(mean_absolute_error(y_val, model.predict(X_val))),
        "val_rmse": float(np.sqrt(mean_squared_error(y_val, model.predict(X_val)))),
        "val_r2": float(r2_score(y_val, model.predict(X_val))),
        "test_mae": float(mean_absolute_error(y_test, model.predict(X_test))),
        "test_rmse": float(np.sqrt(mean_squared_error(y_test, model.predict(X_test)))),
        "test_r2": float(r2_score(y_test, model.predict(X_test)))
    }
    
    return metrics, model


In [9]:
results = {"cost_prediction": {}, "co2_prediction": {}}

print("Training COST prediction models...\n")

for name, model in models.items():
    metrics, trained = evaluate_model(
        model, X_train, y_cost_train, X_val, y_cost_val, X_test, y_cost_test
    )
    
    results["cost_prediction"][name] = metrics
    
    joblib.dump(trained, f"{MODELS_PATH}cost_{name.replace(' ', '_').lower()}.pkl")
    
    print(f"{name}: Test R² = {metrics['test_r2']:.4f}")


Training COST prediction models...

Linear Regression: Test R² = 1.0000
Ridge Regression: Test R² = 1.0000
Decision Tree: Test R² = 1.0000
Random Forest: Test R² = 1.0000
XGBoost: Test R² = 1.0000


In [10]:
print("\nTraining CO₂ prediction models...\n")

for name in models.keys():
    model = models[name].__class__(**models[name].get_params())
    
    metrics, trained = evaluate_model(
        model, X_train, y_co2_train, X_val, y_co2_val, X_test, y_co2_test
    )
    
    results["co2_prediction"][name] = metrics
    
    joblib.dump(trained, f"{MODELS_PATH}co2_{name.replace(' ', '_').lower()}.pkl")
    
    print(f"{name}: Test R² = {metrics['test_r2']:.4f}")



Training CO₂ prediction models...

Linear Regression: Test R² = 1.0000
Ridge Regression: Test R² = 1.0000
Decision Tree: Test R² = 1.0000
Random Forest: Test R² = 1.0000
XGBoost: Test R² = 1.0000


In [11]:
pd.DataFrame(results["cost_prediction"]).T.to_csv(
    f"{METRICS_PATH}baseline_metrics_cost.csv"
)
pd.DataFrame(results["co2_prediction"]).T.to_csv(
    f"{METRICS_PATH}baseline_metrics_co2.csv"
)

with open(f"{METRICS_PATH}baseline_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("✓ Metrics saved")


✓ Metrics saved


In [12]:
best_cost = max(results["cost_prediction"].items(), key=lambda x: x[1]["test_r2"])
best_co2  = max(results["co2_prediction"].items(), key=lambda x: x[1]["test_r2"])

print("Best Cost Model:", best_cost[0], "| R²:", best_cost[1]["test_r2"])
print("Best CO₂ Model :", best_co2[0], "| R²:", best_co2[1]["test_r2"])


Best Cost Model: Linear Regression | R²: 1.0
Best CO₂ Model : Linear Regression | R²: 1.0
